# 07 - Une image vraiment inconnue : les inondations d'Émilie-Romagne (mai 2023)

Jusqu'ici, toutes les images venaient de Sen1Floods11 (2016-2019). Ici, une image **totalement
nouvelle** pour le système :

- **événement** : crues des 16 et 17 mai 2023 en Émilie-Romagne (Italie), autour de Lugo et
  Conselice (province de Ravenne). L'Italie n'est pas dans Sen1Floods11 ;
- **images** : Sentinel-2 L1C brutes, telles que distribuées par le programme Copernicus
  (copie publique de Google Cloud, `gcp-public-data-sentinel-2`), tuile T32TQQ :
  **23 avril 2023** (avant) et **23 mai 2023** (après, même orbite) ;
- **pas d'annotation** : on compare avec ce qu'on voit sur l'image et avec les cartes du service
  Copernicus de cartographie d'urgence (activation EMSR664).

Deux différences techniques avec Sen1Floods11, gérées ci-dessous :
1. depuis 2022, les valeurs L1C ont un **décalage de +1000** (*processing baseline* 04.00) : on le
   retire pour retrouver l'échelle des images d'entraînement ;
2. les bandes à 20 m et 60 m sont ramenées à 10 m (plus proche voisin, comme Sen1Floods11).

Exécution : Colab, *Exécution > Tout exécuter* (environ 3 minutes, pas besoin de GPU).

In [ ]:
import os, re, glob, subprocess, sys

if not os.path.isdir("one-look"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lemembre05/one-look.git"], check=True)
else:
    subprocess.run(["git", "-C", "one-look", "pull", "-q"], check=True)
os.chdir("one-look")
sys.path.insert(0, ".")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "onnxruntime", "pyproj"], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, rasterio
from rasterio.windows import Window
from pyproj import Transformer
from src import demo, donnees

systeme = demo.OneLook("modeles")
print("Seuils :", systeme.seuils)

## 1. Paramètres

Zone de 20 km × 20 km (2048 × 2048 pixels de 10 m) centrée entre Lugo et Conselice, découpée
ensuite en 16 vignettes de 512 × 512 pixels, la taille des images d'entraînement.

In [ ]:
BUCKET = "gs://gcp-public-data-sentinel-2/tiles/32/T/QQ"
DATES = {"avant (23 avril 2023)": "20230423T100551", "après (23 mai 2023)": "20230523T100601"}
CENTRE = (44.46, 11.88)      # latitude, longitude
TAILLE = 2048                # pixels de 10 m
VIGNETTE = 512
ORDRE = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12"]
RESOLUTION = {"B01": 60, "B02": 10, "B03": 10, "B04": 10, "B05": 20, "B06": 20, "B07": 20, "B08": 10,
              "B8A": 20, "B09": 60, "B10": 60, "B11": 20, "B12": 20}

## 2. Téléchargement et lecture des images

In [ ]:
def trouver_produit(date):
    r = subprocess.run(f"gsutil ls {BUCKET}/ | grep MSIL1C_{date}", shell=True, capture_output=True, text=True)
    produits = sorted(l.strip().rstrip("/") for l in r.stdout.splitlines() if l.strip())
    if not produits:
        raise RuntimeError(f"aucun produit pour {date}")
    return produits[-1]                      # version la plus récente si retraitée

def telecharger(date):
    dossier = f"s2_{date}"
    if not glob.glob(f"{dossier}/*_B12.jp2"):
        os.makedirs(dossier, exist_ok=True)
        produit = trouver_produit(date)
        subprocess.run(f'gsutil -m -q cp "{produit}/GRANULE/*/IMG_DATA/*_B*.jp2" {dossier}/', shell=True, check=True)
        subprocess.run(f'gsutil -q cp "{produit}/MTD_MSIL1C.xml" {dossier}/', shell=True, check=True)
    return dossier

def decalage(dossier):
    """Décalage radiométrique L1C (-1000 depuis 2022, 0 avant), lu dans les métadonnées."""
    chemin = f"{dossier}/MTD_MSIL1C.xml"
    if not os.path.exists(chemin):
        return 0
    m = re.findall(r"<RADIO_ADD_OFFSET[^>]*>(-?\d+)</RADIO_ADD_OFFSET>", open(chemin).read())
    return int(m[0]) if m else 0

def lire_zone(dossier, centre=CENTRE, taille=TAILLE):
    """Image (13, taille, taille) à 10 m, dans l'ordre et à l'échelle de Sen1Floods11."""
    ref = glob.glob(f"{dossier}/*_B02.jp2")[0]
    with rasterio.open(ref) as f:
        crs, transform10 = f.crs, f.transform
    x, y = Transformer.from_crs("EPSG:4326", crs, always_xy=True).transform(centre[1], centre[0])
    ligne, col = rasterio.transform.rowcol(transform10, x, y)
    l0, c0 = ligne - taille // 2, col - taille // 2
    off = decalage(dossier)
    bandes = []
    for b in ORDRE:
        k = RESOLUTION[b] // 10
        with rasterio.open(glob.glob(f"{dossier}/*_{b}.jp2")[0]) as f:
            n = -(-taille // k) + 1
            a = f.read(1, window=Window(c0 // k, l0 // k, n, n), boundless=True, fill_value=0)
        a = np.repeat(np.repeat(a, k, axis=0), k, axis=1)
        a = a[l0 - (l0 // k) * k:][:taille, c0 - (c0 // k) * k:][:, :taille].astype(np.float32)
        bandes.append(np.where(a > 0, np.clip(a + off, 1, None), 0))
    zone_transform = rasterio.transform.Affine(transform10.a, 0, transform10.c + c0 * transform10.a,
                                               0, transform10.e, transform10.f + l0 * transform10.e)
    return np.stack(bandes), crs, zone_transform, off

images = {}
for nom, date in DATES.items():
    dossier = telecharger(date)
    s2, crs, zt, off = lire_zone(dossier)
    images[nom] = s2
    print(f"{nom} : {s2.shape}, décalage radiométrique {off}, réflectance médiane B02 {np.median(s2[1]):.0f}")

## 3. Eau permanente

Le système a besoin de la carte de l'eau permanente (rivières, lacs, canaux) pour ne pas la
confondre avec une inondation. On utilise la même source que Sen1Floods11 : le *Global Surface
Water* du Centre commun de recherche européen (JRC), en gardant les pixels en eau au moins 90 %
du temps entre 1984 et 2021. Si le téléchargement échoue, l'analyse continue sans cette carte.

In [ ]:
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

def eau_permanente(crs, transform, taille=TAILLE, seuil=90):
    url = "/vsicurl/https://storage.googleapis.com/global-surface-water/downloads2021/occurrence/occurrence_10E_50Nv1_4_2021.tif"
    try:
        with rasterio.open(url) as src, WarpedVRT(src, crs=crs, transform=transform, width=taille, height=taille,
                                                  resampling=Resampling.nearest) as vrt:
            occ = vrt.read(1)
        return ((occ >= seuil) & (occ <= 100)).astype(np.uint8)
    except Exception as e:
        print("carte d'eau permanente indisponible :", e)
        return None

perm = eau_permanente(crs, zt)
if perm is not None:
    print(f"eau permanente : {perm.mean():.2%} de la zone")

## 4. Le système sur toute la zone (20 km × 20 km)

In [ ]:
resultats = {}
for nom, s2 in images.items():
    res = systeme.analyser(s2, perm)
    resultats[nom] = res
    demo.figure(s2, res, nom); plt.show()
    print(demo.resume(res), "\n")

## 5. Vignette par vignette

À bord, chaque vignette de 5 km × 5 km reçoit sa propre décision : seules les vignettes en
ALERTE seraient envoyées au sol en priorité.

In [ ]:
COULEURS = demo.COULEURS
n = TAILLE // VIGNETTE
lignes = []
fig, axes = plt.subplots(1, len(images), figsize=(8 * len(images), 8))
for ax, (nom, s2) in zip(np.atleast_1d(axes), images.items()):
    ax.imshow(donnees.rgb(s2)); ax.set_title(nom); ax.axis("off")
    for i in range(n):
        for j in range(n):
            sl = (slice(i * VIGNETTE, (i + 1) * VIGNETTE), slice(j * VIGNETTE, (j + 1) * VIGNETTE))
            r = systeme.analyser(s2[(slice(None),) + sl], None if perm is None else perm[sl])
            lignes.append({"date": nom, "vignette": f"{i},{j}", "decision": r["decision"],
                           "eau": round(r["part_eau"], 3), "nuages": round(r["nuages"], 2),
                           "score_inconnu": round(r["score_inconnu"], 1)})
            ax.add_patch(plt.Rectangle((j * VIGNETTE + 8, i * VIGNETTE + 8), VIGNETTE - 16, VIGNETTE - 16,
                                       fill=False, lw=3, ec=COULEURS[r["decision"]]))
            ax.text(j * VIGNETTE + 20, i * VIGNETTE + 60, r["decision"], color="white", fontsize=9,
                    bbox=dict(facecolor=COULEURS[r["decision"]], alpha=0.8, pad=2))
plt.tight_layout(); plt.show()
vignettes = pd.DataFrame(lignes)
print(vignettes.pivot_table(index="date", columns="decision", values="vignette", aggfunc="count", fill_value=0))

## 6. Garder l'image pour l'application

La zone « après » est enregistrée dans `exemples/` au format des images d'entraînement
(GeoTIFF à 13 bandes) : l'application de démonstration la proposera en exemple.

In [ ]:
os.makedirs("exemples", exist_ok=True)
for nom, cle in [("après (23 mai 2023)", "Italie_20230523"), ("avant (23 avril 2023)", "Italie_20230423")]:
    s2 = images[nom][:, :VIGNETTE * 2, :VIGNETTE * 2]          # 10 km x 10 km, coin nord-ouest
    profil = dict(driver="GTiff", height=s2.shape[1], width=s2.shape[2], count=13, dtype="int16",
                  crs=crs, transform=zt, compress="deflate")
    with rasterio.open(f"exemples/{cle}_S2Hand.tif", "w", **profil) as f:
        f.write(s2.astype(np.int16))
    if perm is not None:
        with rasterio.open(f"exemples/{cle}_JRCWaterHand.tif", "w", **{**profil, "count": 1, "dtype": "uint8"}) as f:
            f.write(perm[None, :VIGNETTE * 2, :VIGNETTE * 2])
print(sorted(os.listdir("exemples")))